# Customer Support Intent Classifier

This notebook trains and evaluates a TF-IDF plus Logistic Regression classifier using the supplied customer-support dataset.

The dataset is an intent-classification dataset, not learner-course interaction data for collaborative filtering.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

root = Path.cwd()
candidates = [root / 'data' / 'aiml_training_data.csv', Path(r'C:/Users/supre/Downloads/aiml_training_data.csv')]
data_path = next(path for path in candidates if path.exists())
df = pd.read_csv(data_path)
df.head()

In [ ]:
print(df.shape)
print(df.isna().sum())
display(df['intent'].value_counts())
sns.countplot(data=df, y='intent', order=df['intent'].value_counts().index)
plt.title('Intent distribution')
plt.show()

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    df['text'], df['intent'], test_size=0.2, random_state=42, stratify=df['intent']
)
model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
])
model.fit(x_train, y_train)
predictions = model.predict(x_test)
print(f'Accuracy: {accuracy_score(y_test, predictions):.3f}')
print(classification_report(y_test, predictions, zero_division=0))

In [ ]:
labels = sorted(df['intent'].unique())
matrix = confusion_matrix(y_test, predictions, labels=labels)
sns.heatmap(matrix, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
messages = [
    'Where is my order?',
    'I need to send this item back',
    'The product arrived damaged',
]
for message, intent in zip(messages, model.predict(messages)):
    print(f'{message} -> {intent}')